# LRTIA — Long-Range Token Influence Analysis

**A walkthrough of the four-condition method.**

This notebook teaches the core method behind the LRTIA project: a way to measure how much earlier text predicts later text in human language, and how that influence decays as a function of distance.

By the end of this notebook you will:

1. Understand what we mean by "influence" of prior context on a target span (it's a perplexity difference).
2. See how the **four conditions** — ordered, sentence-shuffled, sentence-reversed, token-shuffled — together isolate three distinct kinds of structure in language.
3. Run the full pipeline end-to-end on a small public corpus with a small probe model (distilgpt2). You can hit *Run All* and the notebook will produce the canonical four-curve plot.

This notebook uses distilgpt2 so it runs in a few minutes on free Colab or a laptop CPU. The numbers will differ from the Llama-3.1-8B results in the paper, but the qualitative shape is the same — the method is what matters.

---


## 1. What we're trying to measure

Language unfolds in time. Each sentence is followed by another. Each word is followed by another. The question we're asking is:

> *How much does what was said earlier predict what comes next, as a function of how far back you go?*

To answer this, we pick a short **target** span of text (say, 30 tokens deep into a document) and ask: how surprised is a language model by that target?

If we give the model *zero* prior context, the model has to predict the target cold — it'll be very surprised (high perplexity). If we give it *all* the prior context, it should be much less surprised (low perplexity, because the prior context narrowed things down). The difference is a measure of how much prior context helps.

But we can do something more interesting: give the model more and more context, in increments. Plot how perplexity falls as a function of distance. The shape of that curve tells us about the **structure** of long-range influence in the text.

### Why four conditions?

A single "intact-context vs. no-context" comparison conflates several different things. Imagine I give the model the previous 100 tokens of a story. Some of the predictive value comes from:

- **Sequential structure** (sentence 2 elaborates sentence 1, and the model uses that)
- **Coherent content** (the tokens that appeared before are about the same topic, so the model expects similar topical vocabulary in the target)
- **Direction** (events unfold in forward time, and the next event tends to follow from the previous one)

To pull these apart, we run four versions of the prior context:

| Condition | What it preserves | What it destroys |
|-----------|-------------------|------------------|
| `ordered` | everything: order, content, direction | nothing |
| `sentence_shuffled` | sentence-internal syntax, content | sentence order |
| `sentence_reversed` | sentence-internal syntax, content, adjacency | forward direction |
| `token_shuffled` | tokens themselves | order, content structure, sentences |

By comparing these four against each other, we get three diagnostic contrasts:

- **Direction-specific** influence  =  `ordered` − `sentence_reversed`  (what's lost when you flip the direction)
- **Content-driven** influence  =  `sentence_shuffled` − `token_shuffled`  (what coherent-but-scrambled-content adds over random tokens)
- **Sequence-permutation** influence  =  `ordered` − `sentence_shuffled`  (what specific sentence order adds over scrambled order)

Each contrast isolates a different aspect of what makes prior context useful.


## 2. Setup

We use:
- `transformers` for the probe model
- `datasets` for a small public corpus (wikitext-2)
- `torch`, `numpy`, `matplotlib` for math and plotting

If you're on Colab the next cell will install what's missing.


In [ ]:
# Install dependencies if needed (Colab has most of these already).
import sys, subprocess
def pip_install(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

try:
    import transformers, datasets  # noqa: F401
except ImportError:
    pip_install('transformers', 'datasets', 'accelerate')

import json, math, random, re
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer

SEED = 20260528
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')


## 3. Load the probe model

We use **distilgpt2** — a small (~82M parameter) autoregressive language model. It's tiny by modern standards, but it's:

- Fast enough to run on CPU in a few minutes
- Large enough to pick up the qualitative structure we care about
- Public, free, no auth required

In the actual paper we use Llama-3.1-8B and Mistral-7B. Replace `MODEL_NAME` below if you have a bigger GPU.


In [ ]:
MODEL_NAME = 'distilgpt2'  # try 'gpt2' for slightly better signal, or 'gpt2-medium' if you have a GPU

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()

# Context limit for the model — we won't try to feed more than this.
MODEL_MAX_CTX = model.config.n_positions
print(f'Loaded {MODEL_NAME}, max context = {MODEL_MAX_CTX} tokens')


## 4. Perplexity of a target span

The core measurement is: **given some context tokens, how surprised is the model by a fixed target span?**

We compute the per-token negative log-likelihood (NLL) of the target, then exponentiate to get perplexity. Lower perplexity = more predictable target = the context was helpful.

The function below takes a list of context token IDs and a list of target token IDs, concatenates them, runs the model, and computes the average NLL over just the target positions.

A subtle point: when the model sees context+target, it predicts each target token from the tokens that came before it (which includes the context plus the earlier-target tokens). So this measures the conditional probability of the target *given the context*.


In [ ]:
@torch.no_grad()
def ppl_at_target(ctx_ids, tgt_ids):
    """Compute per-token perplexity of `tgt_ids` conditioned on `ctx_ids`.

    Both are 1-D lists/arrays of integer token IDs. Returns a scalar perplexity.
    """
    # Concatenate context + target.
    all_ids = list(ctx_ids) + list(tgt_ids)
    # Truncate from the left if we'd exceed the model's context window.
    if len(all_ids) > MODEL_MAX_CTX:
        all_ids = all_ids[-MODEL_MAX_CTX:]
    input_ids = torch.tensor([all_ids], device=DEVICE)

    # Run the model.
    logits = model(input_ids).logits  # shape: [1, seq_len, vocab]

    # The model predicts token i+1 from tokens 0..i, so we shift.
    shift_logits = logits[:, :-1, :]
    shift_labels = input_ids[:, 1:]
    log_probs = torch.log_softmax(shift_logits.float(), dim=-1)
    # Gather the log-prob of the actual next token at each position.
    token_log_probs = log_probs.gather(-1, shift_labels.unsqueeze(-1)).squeeze(-1)  # [1, seq_len-1]

    # We only want the NLL at *target* positions.
    # In the shifted sequence, target tokens occupy the last len(tgt_ids) positions.
    target_nll = -token_log_probs[0, -len(tgt_ids):]
    mean_nll = target_nll.mean().item()
    return math.exp(mean_nll)

# Quick sanity check:
ctx = tokenizer.encode('The quick brown fox jumps over the lazy', add_special_tokens=False)
tgt = tokenizer.encode(' dog.', add_special_tokens=False)
print(f'ppl(target | context) = {ppl_at_target(ctx, tgt):.2f}')
print(f'ppl(target | nothing) = {ppl_at_target([], tgt):.2f}')


## 5. Building the four conditions

For each target, we need to construct four versions of the prior context:

1. `ordered` — the actual preceding text, unchanged
2. `sentence_shuffled` — same sentences, random order
3. `sentence_reversed` — same sentences, reversed order (last sentence first)
4. `token_shuffled` — same tokens, random order (no sentence structure)

**Key invariant:** all four conditions use *exactly the same set of tokens*. They differ only in how those tokens are arranged. This is what makes the contrasts interpretable — if `ordered` predicts the target better than `sentence_reversed`, the only thing that can explain it is the *order* (because the tokens are identical).

To preserve that invariant, we select prior context by taking whole trailing sentences whose combined length is at most `n_ctx_tokens`, then manipulate the order of those exact sentences. We need a sentence splitter to do this. A regex-based splitter is good enough for English-language plain text. (In the paper we use the same approach.)


In [ ]:
SENT_BOUNDARY = re.compile(r'(?<=[.!?])\s+(?=[A-Z])')

def split_sentences(text):
    """Split text into sentences using a simple regex. Good enough for English."""
    sents = SENT_BOUNDARY.split(text.strip())
    return [s.strip() for s in sents if s.strip()]


def build_four_contexts(prior_text, n_ctx_tokens, rng):
    """Build the four condition versions of a prior context.

    `ordered` and `token_shuffled` are computed from the last n_ctx_tokens
    of the flat prior, so they're always defined at every context length.

    `sentence_shuffled` and `sentence_reversed` are computed from a set of
    trailing sentences whose combined token count is <= n_ctx_tokens — they
    use *exactly the same set of tokens* as each other (so the comparison is
    clean), but may differ slightly in token count from `ordered`/`token_shuffled`
    at small context lengths where a sentence boundary doesn't line up.
    They return None when we can't fit at least 2 sentences in the window.

    Returns a dict mapping condition name -> list of token IDs (or None).
    """
    # Always-available conditions: ordered and token_shuffled, using the
    # last n_ctx_tokens of the flat prior text.
    full_ids = tokenizer.encode(prior_text, add_special_tokens=False)
    flat_tail = full_ids[-n_ctx_tokens:]
    ordered = list(flat_tail)
    tok_shuffled = list(flat_tail)
    rng.shuffle(tok_shuffled)

    # Sentence-level conditions: only if we have >= 2 trailing sentences that fit.
    sentences = split_sentences(prior_text)
    sent_shuffled = None
    sent_reversed = None

    if len(sentences) >= 2:
        sent_tokens = [tokenizer.encode(s + ' ', add_special_tokens=False) for s in sentences]
        # Walk backward through sentences, collecting until we'd exceed n_ctx_tokens.
        selected = []
        total = 0
        for s_toks in reversed(sent_tokens):
            if total + len(s_toks) > n_ctx_tokens and selected:
                break
            selected.append(s_toks)
            total += len(s_toks)
        selected.reverse()

        if len(selected) >= 2:
            # Permute sentences for sent_shuffled.
            shuf_idx = list(range(len(selected)))
            rng.shuffle(shuf_idx)
            sent_shuffled = [t for i in shuf_idx for t in selected[i]]
            # Reverse sentences for sent_reversed.
            sent_reversed = [t for s in reversed(selected) for t in s]

    return {
        'ordered': ordered,
        'sentence_shuffled': sent_shuffled,
        'sentence_reversed': sent_reversed,
        'token_shuffled': tok_shuffled,
    }


## 6. Compute curves for a single document

The protocol per document:

1. Pick a **target span** somewhere in the middle of the document (we use the position at 50% by default).
2. The target is a fixed 30-token window.
3. The **prior context** is everything that comes before the target in the document.
4. We measure perplexity of the target at a set of context lengths: 0, 1, 2, 4, 8, ..., 512.
5. At each context length, we construct all four conditions and measure perplexity for each.

This gives us four curves per document. We aggregate across documents to get the final figure.


In [ ]:
CTX_LENGTHS = [0, 1, 2, 4, 8, 16, 32, 64, 128, 256, 512]
TARGET_LEN = 30
TARGET_FRAC = 0.5  # take target at midpoint of document
MIN_DOC_TOKENS = max(CTX_LENGTHS) + TARGET_LEN + 50  # need enough room


def four_condition_curves(text, rng):
    """Run the four conditions at each context length on a single document.

    Returns a dict mapping condition -> list of perplexities (one per CTX_LENGTHS),
    or None if the document is too short.

    `sentence_shuffled` and `sentence_reversed` entries may be NaN at small
    context lengths where there aren't enough sentences to manipulate.
    """
    full_ids = tokenizer.encode(text, add_special_tokens=False)
    if len(full_ids) < MIN_DOC_TOKENS:
        return None

    tgt_start = int(len(full_ids) * TARGET_FRAC)
    tgt_end = tgt_start + TARGET_LEN
    if tgt_end > len(full_ids):
        return None
    tgt_ids = full_ids[tgt_start:tgt_end]

    prior_ids = full_ids[:tgt_start]
    prior_text = tokenizer.decode(prior_ids)

    out = {cond: [] for cond in ['ordered', 'sentence_shuffled',
                                  'sentence_reversed', 'token_shuffled']}
    for c in CTX_LENGTHS:
        if c == 0:
            # Zero-context baseline: all four conditions are identical.
            p = ppl_at_target([], tgt_ids)
            for cond in out:
                out[cond].append(p)
            continue

        four = build_four_contexts(prior_text, c, rng)
        for cond, ctx_ids in four.items():
            if ctx_ids is None:
                out[cond].append(float('nan'))
            else:
                out[cond].append(ppl_at_target(ctx_ids, tgt_ids))

    return out


## 7. Get a small public corpus

We use **wikitext-2**, a small slice of English Wikipedia, available through HuggingFace `datasets`. It's a few hundred articles, of varying lengths.

For teaching purposes we use ~10 articles. The paper uses 60 documents per corpus across 8 languages. The qualitative pattern is visible with very few documents; scale gives you tighter error bars.


In [ ]:
from datasets import load_dataset

# wikitext-2 raw — small, well-formed English text.
ds = load_dataset('wikitext', 'wikitext-2-raw-v1', split='train')

# wikitext-2 stores text in line-chunks. Reassemble into articles using the
# '= Article Title =' headers that mark article boundaries in wikitext.
def reassemble_articles(ds, min_chars=2000, max_articles=10):
    articles = []
    current = []
    for row in ds:
        line = row['text']
        # Top-level header lines look like '= Title =\n'
        if re.match(r'^\s*=\s[^=].*?\s=\s*$', line) and current:
            joined = ''.join(current).strip()
            if len(joined) >= min_chars:
                articles.append(joined)
                if len(articles) >= max_articles:
                    return articles
            current = []
        current.append(line)
    if current:
        joined = ''.join(current).strip()
        if len(joined) >= min_chars:
            articles.append(joined)
    return articles

docs = reassemble_articles(ds, min_chars=2500, max_articles=10)
print(f'Loaded {len(docs)} documents')
print(f'First doc preview: {docs[0][:200]}...')


## 8. Run the experiment

For each document, compute the four condition curves. This is the heaviest cell — it runs the model many times per document. On distilgpt2 + CPU, it's a few minutes for 10 documents.

Each document yields one curve per condition. We collect them all into a list and aggregate next.


In [ ]:
from tqdm.auto import tqdm

rng = random.Random(SEED)
all_results = []
for i, text in enumerate(tqdm(docs, desc='docs')):
    r = four_condition_curves(text, rng)
    if r is None:
        continue
    all_results.append(r)

print(f'Got curves for {len(all_results)} of {len(docs)} documents')


## 9. Aggregate and plot

For each condition and each context length, we take the mean across documents. We plot perplexity (log scale) against context length (log scale).

In the resulting figure you should see:

- **`ordered`** is the lowest curve — actual forward context helps most.
- **`token_shuffled`** is the highest curve — random tokens are useless.
- **`sentence_shuffled`** and **`sentence_reversed`** sit in between, both above `ordered` and below `token_shuffled`.
- At long context lengths, `sentence_reversed` should be *higher* (worse) than `sentence_shuffled`. This is the headline finding of the paper: reversing direction costs more than shuffling order, because forward direction is doing work above and beyond "having some sequence."


In [ ]:
# Build a [n_docs x n_ctx] array per condition, then take nanmean.
def aggregate(results, cond):
    arr = np.array([r[cond] for r in results])
    return np.nanmean(arr, axis=0), np.nanstd(arr, axis=0) / max(1, np.sqrt(arr.shape[0]))

conditions = ['ordered', 'sentence_shuffled', 'sentence_reversed', 'token_shuffled']
colors = {'ordered': '#1f77b4', 'sentence_shuffled': '#ff7f0e',
          'sentence_reversed': '#d62728', 'token_shuffled': '#7f7f7f'}

fig, ax = plt.subplots(figsize=(8, 5))
xs = np.array(CTX_LENGTHS)
xs_plot = np.where(xs == 0, 0.5, xs)  # avoid log(0) — display 0 as 0.5

for cond in conditions:
    m, se = aggregate(all_results, cond)
    ax.plot(xs_plot, m, marker='o', color=colors[cond], label=cond)
    ax.fill_between(xs_plot, m - se, m + se, color=colors[cond], alpha=0.15)

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Context length (tokens)')
ax.set_ylabel('Target perplexity')
ax.set_title(f'Four-condition LRTIA curves  ({MODEL_NAME}, {len(all_results)} docs)')
ax.legend()
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()


## 10. How to read the plot, and where to go next

### The three diagnostic contrasts

At each context length you can read off three quantities from the four curves:

| Contrast | What it isolates | Where to look |
|----------|------------------|---------------|
| `ordered` vs. `sentence_reversed` | **Direction-specific** influence | gap between the lowest and the reversed curve |
| `sentence_shuffled` vs. `token_shuffled` | **Content-driven** influence | gap between the orange and gray curves |
| `ordered` vs. `sentence_shuffled` | **Sequence-permutation** influence | gap between the lowest and the shuffled curve |

The most important comparison in the paper is `ordered` vs `sentence_reversed` at long range. Reversing destroys forward direction while preserving every other property of the prior context (same tokens, same adjacency, same sentence-internal syntax). If the gap between these two curves grows with distance, that's evidence that **forward sequential direction itself** is what carries long-range coherence — not just "having coherent content nearby."

### Things to try

1. **More documents.** Set `max_articles` higher in §7 and rerun. Curves should smooth out.
2. **A bigger model.** Switch `MODEL_NAME` to `'gpt2'` or `'gpt2-medium'` in §3 (needs a GPU for medium). The qualitative pattern should be similar, the absolute perplexities lower.
3. **Multiple targets per document.** Currently we use one target at position 0.5. The paper uses three: 0.25, 0.5, 0.75. Adapt `four_condition_curves` to loop over `TARGET_FRAC`.
4. **Multiple shuffle seeds.** Each shuffle is one random draw. Average over several seeds for tighter estimates of the shuffled and reversed curves.
5. **A different corpus.** Try a fiction sample, a news article, or a spoken-language transcript. The shape of the curves changes by genre.
6. **The corrected marginal.** Instead of plotting perplexities directly, compute `Δ_d = m_ordered_d − m_token_shuffled_d` where `m_d` is the per-token marginal perplexity reduction at distance `d`. This is the version of the curve used in the main paper figures — it controls for distributional calibration.

### Where to read further

- The paper draft and figures live in the `paper/` folder of the project.
- The analysis scripts in `analysis/` (especially `sentshuffle_decomposition.py`) show the corrected-marginal computation in full.
- The full-scale experiment notebooks (`Corpus_Expansion_LongRange_*.ipynb`) use the same method scaled to multiple languages and the Llama-3.1-8B probe.

Happy probing.
